# Анализ RTT

Рассчет проводился при следующих условиях:

* Сервер - на ноутбуке, подключенный через Wi-Fi
* Клиент - на стационарном компьютере, подключенный через Ethernet кабель к Wi-Fi роутеру
* Количество отправленных пакетов в прогоне - 100
* Прогон производился при данных интервалах
  * 80000000
  * 90000000
  * 100000000
  * 110000000
  * 120000000
  * 130000000
* Каждый прогон проводился только после перезапуска сервера

In [ ]:
! pip install plotly numpy pandas
! pip install --upgrade nbformat
! pip install --upgrade kaleido

In [1]:
import plotly.express as px
import numpy as np
import pandas as pd

In [2]:
intervals = [
    80000000,
    90000000,
    100000000,
    110000000,
    120000000,
    130000000,
]
ATTEMPTS = 100

## График RTT от попытки отправки пакета

In [3]:
dfs = []
for interval in intervals:
    df = pd.read_csv(f"./data/{interval}ns.csv")
    dfs.append(df)

df = pd.concat(dfs)
fig = px.line(
    df,
    x=df.index,
    y="rtt",
    color="interval",
    title=f"RTT to attempt with interval - {interval} ns")
fig.write_image("images/rtt_to_attempt.png", width=900, height=600)
fig.show()

## График среднего значения RTT на всех попытках до

In [4]:
dfs = []
for interval in intervals:
    df = pd.read_csv(f"./data/{interval}ns.csv")
    df['rtt'] = np.cumsum(df['rtt']) / (1 + np.arange(ATTEMPTS))
    dfs.append(df)

df = pd.concat(dfs)
fig = px.line(
    df,
    x=df.index,
    y="rtt",
    color="interval",
    title=f"RTT to attempt with interval - {interval} ns")
fig.write_image("images/rtt_to_attempt_average_cumulative.png", width=900, height=600)
fig.show()

## График среднего RTT на последних двадцати попытках

In [5]:
SMOOTH_ATTEMPTS = 20

dfs = []

for interval in intervals:
    df = pd.read_csv(f"./data/{interval}ns.csv")
    df['rtt'] = (np.cumsum(df['rtt']) - np.append(np.zeros(SMOOTH_ATTEMPTS), np.cumsum(df['rtt'])[:(ATTEMPTS - SMOOTH_ATTEMPTS)])) / SMOOTH_ATTEMPTS
    dfs.append(df)

df = pd.concat(dfs)
fig = px.line(
    df,
    x=df.index,
    y="rtt",
    color="interval",
    title=f"RTT to attempt with interval - {interval} ns")
fig.write_image(f"images/rtt_to_attempt_average_last_{SMOOTH_ATTEMPTS}.png", width=900, height=600)
fig.show()